# 08 — Тематическое моделирование колонки `topic` в DialogSum-RU

Этот ноутбук — самостоятельный (Jupyter / Google Colab) эксперимент по **unsupervised topic modeling** над колонкой `topic` датасета [`d0rj/dialogsum-ru`](https://huggingface.co/datasets/d0rj/dialogsum-ru).

Сравниваются три подхода:

1. **LDA** (Latent Dirichlet Allocation) поверх `CountVectorizer`.
2. **NMF** (Non-negative Matrix Factorization) поверх `TfidfVectorizer`.
3. **BERTopic** поверх мультиязычных sentence-эмбеддингов (`paraphrase-multilingual-MiniLM-L12-v2`).

Цель — понять, как методы группируют темы диалогов и какой из них даёт наиболее осмысленные кластеры для последующей задачи распознавания интентов.

**Важно:** здесь не обучаются модели классификации интентов — только тематическое моделирование текста `topic`.

**Источник данных (parquet через `hf://`):**
- `train`: `data/train-00000-of-00001-bcc43b46acda4001.parquet`
- `validation`: `data/validation-00000-of-00001-7e263d81db1c7a12.parquet`
- `test`: `data/test-00000-of-00001-2f13615b955ea947.parquet`

## 1. Импорты и настройки

При запуске в Colab при необходимости раскомментируйте установку зависимостей. BERTopic тянет тяжёлые пакеты (`umap-learn`, `hdbscan`, `sentence-transformers`), поэтому ставится отдельно.

In [ ]:
# При необходимости (особенно в Colab) обновите/установите зависимости:
# !pip install -q -U pandas pyarrow huggingface_hub fsspec scikit-learn
# !pip install -q bertopic sentence-transformers umap-learn hdbscan

In [ ]:
import re
import warnings
from typing import List, Sequence

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_TOPICS = 20  # одинаковое число тем для LDA и NMF
BERT_SAMPLE_SIZE = 5000  # верхняя граница выборки для BERTopic

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

## 2. Загрузка данных

Читаем три parquet-сплита напрямую с Hugging Face через `hf://` и склеиваем в один DataFrame `df_all` с дополнительной колонкой `split`.

In [ ]:
HF_BASE = "hf://datasets/d0rj/dialogsum-ru/"
SPLIT_FILES = {
    "train": "data/train-00000-of-00001-bcc43b46acda4001.parquet",
    "validation": "data/validation-00000-of-00001-7e263d81db1c7a12.parquet",
    "test": "data/test-00000-of-00001-2f13615b955ea947.parquet",
}

frames = []
for split_name, rel_path in SPLIT_FILES.items():
    df_split = pd.read_parquet(HF_BASE + rel_path)
    df_split["split"] = split_name
    frames.append(df_split)

df_all = pd.concat(frames, ignore_index=True)
df_all = df_all[["id", "dialogue", "summary", "topic", "split"]]

print("Размер df_all:", df_all.shape)
df_all.head()

In [ ]:
print("Распределение по split:")
print(df_all["split"].value_counts())
print("\nКоличество пропусков в topic:", df_all["topic"].isna().sum())
print("Уникальных значений topic:", df_all["topic"].nunique())

## 3. Предобработка `topic`

Создаём колонку `topic_clean`:
- приводим к нижнему регистру;
- убираем спецсимволы и пунктуацию (оставляем буквы латиницы/кириллицы и цифры);
- схлопываем повторные пробелы;
- пропуски и пустые строки заменяем на пустую строку и при моделировании исключаем.

In [ ]:
_RE_NON_ALNUM = re.compile(r"[^0-9a-zA-Zа-яА-ЯёЁ\s]+", flags=re.UNICODE)
_RE_SPACES = re.compile(r"\s+")


def clean_topic(text) -> str:
    """Безопасная нормализация строки темы."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    s = str(text).lower()
    s = _RE_NON_ALNUM.sub(" ", s)
    s = _RE_SPACES.sub(" ", s).strip()
    return s


df_all["topic_clean"] = df_all["topic"].map(clean_topic)
non_empty_mask = df_all["topic_clean"].str.len() > 0

print("Пустых topic_clean:", (~non_empty_mask).sum())
df_all[["topic", "topic_clean"]].head(10)

In [ ]:
def print_top_terms(model, feature_names: Sequence[str], n_top: int = 10, title: str = "") -> None:
    """Вывод топ-слов на тему для LDA/NMF."""
    if title:
        print(f"\n=== {title} ===")
    for topic_idx, weights in enumerate(model.components_):
        top_idx = weights.argsort()[::-1][:n_top]
        words = [feature_names[i] for i in top_idx]
        print(f"Topic {topic_idx:>2}: {', '.join(words)}")


def assign_dominant_topic(doc_topic_matrix: np.ndarray) -> np.ndarray:
    """Доминирующая тема = argmax по теме."""
    return doc_topic_matrix.argmax(axis=1)


def show_topic_examples(df: pd.DataFrame, topic_col: str, topics: List[int], n_examples: int = 5) -> None:
    """Печать нескольких оригинальных `topic` для выбранных кластеров."""
    for t in topics:
        sub = df.loc[df[topic_col] == t, "topic"].dropna().unique()
        print(f"\n--- {topic_col} = {t} (уникальных тем: {len(sub)}) ---")
        for ex in list(sub)[:n_examples]:
            print(" •", ex)

## 4. LDA (CountVectorizer + LatentDirichletAllocation)

Базовый вероятностный подход. Считаем мешок слов и подгоняем LDA с фиксированным числом тем `N_TOPICS`.

In [ ]:
docs = df_all["topic_clean"].tolist()
docs_for_fit = [d if d else " " for d in docs]  # CountVectorizer не любит пустые строки

count_vectorizer = CountVectorizer(
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b\w\w+\b",
)
X_counts = count_vectorizer.fit_transform(docs_for_fit)
count_features = count_vectorizer.get_feature_names_out()
print("Матрица счётчиков:", X_counts.shape)

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    learning_method="batch",
    max_iter=20,
    random_state=RANDOM_STATE,
)
lda_doc_topic = lda_model.fit_transform(X_counts)
print("Распределение doc-topic:", lda_doc_topic.shape)

In [ ]:
print_top_terms(lda_model, count_features, n_top=10, title="LDA: топ-слова на тему")

In [ ]:
df_all["lda_topic"] = assign_dominant_topic(lda_doc_topic)
# Документам с пустым topic_clean выставим -1, чтобы они не путали интерпретацию
df_all.loc[~non_empty_mask, "lda_topic"] = -1

print("Распределение по LDA-кластерам:")
print(df_all["lda_topic"].value_counts().sort_index())

In [ ]:
lda_top5 = (
    df_all.loc[df_all["lda_topic"] >= 0, "lda_topic"]
    .value_counts()
    .head(5)
    .index.tolist()
)
show_topic_examples(df_all, "lda_topic", lda_top5, n_examples=5)

## 5. NMF (TfidfVectorizer + NMF)

NMF поверх TF-IDF обычно даёт более «острые» темы, чем LDA на коротких текстах.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b\w\w+\b",
)
X_tfidf = tfidf_vectorizer.fit_transform(docs_for_fit)
tfidf_features = tfidf_vectorizer.get_feature_names_out()
print("Матрица TF-IDF:", X_tfidf.shape)

nmf_model = NMF(
    n_components=N_TOPICS,
    init="nndsvda",
    max_iter=400,
    random_state=RANDOM_STATE,
)
nmf_doc_topic = nmf_model.fit_transform(X_tfidf)
print("Распределение doc-topic:", nmf_doc_topic.shape)

In [ ]:
print_top_terms(nmf_model, tfidf_features, n_top=10, title="NMF: топ-слова на тему")

In [ ]:
df_all["nmf_topic"] = assign_dominant_topic(nmf_doc_topic)
df_all.loc[~non_empty_mask, "nmf_topic"] = -1

print("Распределение по NMF-кластерам:")
print(df_all["nmf_topic"].value_counts().sort_index())

In [ ]:
nmf_top5 = (
    df_all.loc[df_all["nmf_topic"] >= 0, "nmf_topic"]
    .value_counts()
    .head(5)
    .index.tolist()
)
show_topic_examples(df_all, "nmf_topic", nmf_top5, n_examples=5)

## 6. BERTopic (sentence embeddings)

Используем мультиязычную модель `paraphrase-multilingual-MiniLM-L12-v2` и BERTopic с настройками по умолчанию (UMAP + HDBSCAN).

Чтобы не перегружать Colab, ограничиваем размер выборки до `min(len(df_all), 5000)`. Если датасет короче — берём всё. Документам, не попавшим в выборку, проставляется метка `-999` (читай: «не моделировалось BERTopic»).

Если BERTopic не установлен, раскомментируйте установку в первой ячейке ноутбука.

In [ ]:
df_all["bert_topic"] = -999  # дефолт: документ не моделировался BERTopic

try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer

    candidate_idx = df_all.index[non_empty_mask].tolist()
    sample_size = min(len(candidate_idx), BERT_SAMPLE_SIZE)

    rng = np.random.RandomState(RANDOM_STATE)
    if sample_size < len(candidate_idx):
        sampled_idx = rng.choice(candidate_idx, size=sample_size, replace=False)
    else:
        sampled_idx = np.array(candidate_idx)

    sampled_idx = sorted(sampled_idx.tolist())
    sampled_docs = df_all.loc[sampled_idx, "topic_clean"].tolist()
    print(f"BERTopic: моделируем {len(sampled_docs)} документов из {len(df_all)}")

    embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    bertopic_model = BERTopic(
        embedding_model=embed_model,
        language="multilingual",
        calculate_probabilities=False,
        verbose=False,
    )
    bert_labels, _ = bertopic_model.fit_transform(sampled_docs)
    bert_labels = np.asarray(bert_labels)

    df_all.loc[sampled_idx, "bert_topic"] = bert_labels
    bertopic_available = True
    print("BERTopic обучен. Уникальных меток (включая -1 = шум):", len(set(bert_labels)))
except Exception as e:
    bertopic_available = False
    bertopic_model = None
    print("BERTopic недоступен или произошла ошибка:", repr(e))
    print("Установите зависимости: pip install bertopic sentence-transformers umap-learn hdbscan")

In [ ]:
if bertopic_available:
    topic_info = bertopic_model.get_topic_info()
    print("Сводка по темам BERTopic (топ-10 по размеру):")
    display(topic_info.head(10))

    print("\nТоп-слова по нескольким темам:")
    for t in topic_info["Topic"].head(10).tolist():
        if t == -1:
            print(f"\nTopic {t} (шум): пропускаем подробный вывод")
            continue
        words = [w for w, _ in bertopic_model.get_topic(t)][:10]
        print(f"Topic {t:>3}: {', '.join(words)}")
else:
    print("Пропускаем вывод тем BERTopic — модель не обучена.")

In [ ]:
print("Распределение по BERTopic (включая -999 = не моделировалось, -1 = шум):")
print(df_all["bert_topic"].value_counts().sort_index().head(20))

if bertopic_available:
    bert_top5 = (
        df_all.loc[df_all["bert_topic"] >= 0, "bert_topic"]
        .value_counts()
        .head(5)
        .index.tolist()
    )
    show_topic_examples(df_all, "bert_topic", bert_top5, n_examples=5)

## 7. Сравнение и интерпретация

Сводная таблица числа кластеров и размера крупнейшего кластера для каждого метода.

In [ ]:
def cluster_summary(series: pd.Series, ignore_values=(-1, -999)) -> dict:
    s = series[~series.isin(ignore_values)]
    counts = s.value_counts()
    return {
        "n_docs": int(s.shape[0]),
        "n_clusters": int(counts.shape[0]),
        "largest_cluster_size": int(counts.iloc[0]) if len(counts) else 0,
        "smallest_cluster_size": int(counts.iloc[-1]) if len(counts) else 0,
    }


summary_rows = {
    "LDA": cluster_summary(df_all["lda_topic"]),
    "NMF": cluster_summary(df_all["nmf_topic"]),
    "BERTopic": cluster_summary(df_all["bert_topic"]),
}
summary_df = pd.DataFrame(summary_rows).T
summary_df

In [ ]:
# Кросс-таблица LDA vs NMF (полезно для оценки взаимного согласия методов)
cross_lda_nmf = pd.crosstab(df_all["lda_topic"], df_all["nmf_topic"])
print("Crosstab LDA × NMF (фрагмент):")
cross_lda_nmf.iloc[:10, :10]

## 8. Выводы (шаблон для заполнения)

После запуска ноутбука заполните пункты ниже на основе полученных распределений и топ-слов. Не выдумываем результаты заранее — формулируем гипотезы, которые проверим по факту.

**8.1. Читаемость и чистота тем: LDA vs NMF**
- LDA на коротких `topic_clean` обычно даёт более «размытые» темы — слова из разных доменов могут смешиваться в одной теме.
- NMF поверх TF-IDF чаще даёт «острые», лексически связанные кластеры (один доминирующий термин и его синонимы).
- _Что наблюдалось здесь:_ <впишите после прогона: насколько слова в топ-10 каждой темы согласованы, есть ли «мусорные» темы>.

**8.2. Группирует ли BERTopic семантически близкие темы лучше?**
- BERTopic работает в пространстве эмбеддингов, поэтому теоретически может объединить перефразы одной темы (например, разные формулировки про «бронирование отеля»).
- Минус — часть документов уходит в шум (`-1`), а на маленькой выборке BERTopic может дробить близкие темы.
- _Что наблюдалось здесь:_ <впишите после прогона: видны ли осмысленные кластеры, какова доля шума>.

**8.3. Какой метод разумно использовать для укрупнения `topic`?**
- Если цель — собрать `topic` в небольшое число интент-подобных макро-классов, и темы короткие и однотипные, NMF/LDA с фиксированным `N_TOPICS` дают предсказуемое число кластеров.
- Если важна семантика и устойчивость к синонимам и переформулировкам, BERTopic скорее предпочтительнее, но требует больше ресурсов и постобработки шума.
- _Итоговая рекомендация для проекта:_ <впишите после прогона: какой метод выбрать и почему>.

**8.4. Дальнейшие шаги**
- Попробовать разное `N_TOPICS` (например, 10/15/20/30) и сравнить размеры/чистоту кластеров.
- Объединить близкие BERTopic-темы вручную или через `reduce_topics`.
- Сопоставить полученные кластеры с целевой схемой интентов из диссертации.